# Simulação de Lançamento da Aurora-Siger

Aluno: Murilo Dias
RM: 575146

Este notebook é uma simulação interativa usando valores de referência de telemetria da nave, usando um processo de decisão GO/NO-GO. E também uma LLM como assistente de bordo chamda Borealis!

Através de sliders e controles interativos, você insere valores de telemetria (temperatura interna e externa, pressão dos tanques, nível de energia, integridade estrutural e status dos módulos críticos). O notebook avalia automaticamente, com base nos valores referenciais de segurança do projeto, se o lançamento pode prosseguir.


️Como rodar: `Ambiente de execução -> Executar tudo`. Depois, é só mudar os valores nos controles do painel e o resultado (checklist e gráficos) atualizará em tempo real.

!!!!!! Para a Borealis, recomendo mudar para uma TPU. E não esqueça de rodar todas as células, tanto para o setup, quanto para o contexto da IA.


## Valores referenciais de segurança

| Parâmetro | Critério de segurança | Observação |
|---|---|---|
| **Temperatura interna** | ≤ 80 °C | Limite de operação das baterias |
| **Temperatura externa** (média últimas 24h) | entre 2 °C e 37 °C | Critério aproximado de lançamento (baseado em diretrizes da NASA); mas na realidade, varia com vento e umidade |
| **Integridade estrutural** | Deve passar na inspeção manual da *ground crew* | 0 = falha estrutural detectada · 1 = passou |
| **Nível de energia** | ≥ 80% da capacidade das baterias | Energia combinada necessária para os ion-thrusters + sistemas da nave até o planeta X |
| **Pressão do tanque LOX/LCH4** | ~325 psi | Valor de referência (nominal); depende do design real da nave |
| **Módulos críticos** | Todos devem estar em status OK | 0 = FALHA · 1 = OK |

### NOTAS IMPORTANTES
- Pressão do tanque: foi adotada uma banda de tolerância de 25 psi em torno dos 325 psi nominais já que o valor exato de tolerância depende do design estrutural do tanque, nozzle etc.
- Energia: o critério dos "80% da capacidade das baterias" foi interpretado como o nível mínimo de carga necessário no momento do lançamento para garantir energia suficiente até o planeta X.
- Temperatura externa: como o critério da NASA é sobre a média das últimas 24h, o notebook simula uma série sintética de telemetria de 24h em torno do valor informado no slider


## Setup

Importa as bibliotecas necessárias


In [1]:
!pip install -q ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 47.4 MB/s eta 0:00:00


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive_output, VBox, HBox, Layout
from IPython.display import display, HTML, clear_output
from datetime import datetime, timedelta

print("OK")

OK


## Valores de referência

Todos os limites de segurança usados na avaliação ficam centralizados aqui, entãs é só editar estes valores para mudá-los.


In [3]:
TEMP_INTERNA_MAX = 80.0
TEMP_EXTERNA_MIN = 2.0
TEMP_EXTERNA_MAX = 37.0
ENERGIA_MINIMA = 80.0
PRESSAO_NOMINAL = 325.0
PRESSAO_TOLERANCIA = 25.0
PRESSAO_MIN = PRESSAO_NOMINAL - PRESSAO_TOLERANCIA
PRESSAO_MAX = PRESSAO_NOMINAL + PRESSAO_TOLERANCIA
MODULOS_CRITICOS = [
    "Propulsão (Ion Thrusters)",
    "Aviônica / Navegação",
    "Suporte à Vida",
    "Comunicações",
    "Sistema Elétrico / Baterias",
]
print("Constantes carregadas:")
print(f"  Temperatura interna máx.: {TEMP_INTERNA_MAX} °C")
print(f"  Temperatura externa (média 24h): {TEMP_EXTERNA_MIN}–{TEMP_EXTERNA_MAX} °C")
print(f"  Energia mínima: {ENERGIA_MINIMA}%")
print(f"  Pressão do tanque: {PRESSAO_MIN}–{PRESSAO_MAX} psi")
print(f"  Módulos monitorados: {len(MODULOS_CRITICOS)}")

Constantes carregadas:
  Temperatura interna máx.: 80.0 °C
  Temperatura externa (média 24h): 2.0–37.0 °C
  Energia mínima: 80.0%
  Pressão do tanque: 300.0–350.0 psi
  Módulos monitorados: 5


## Simulação de valores de telemetria da temperatura externa (24h)

O critério da NASA usa a média das últimas 24h, não uma leitura momentânea. Para simular isso de forma realista, geramos uma série sintética de telemetria em intervalos de 1h em torno do valor atual informado no slider, com uma pequena variação para representar melhor as variações reais de vento, insolação etc.


In [4]:
def simular_telemetria_temp_externa(temp_atual, horas=24, ruido=1.5, seed=42):
    rng = np.random.default_rng(seed)
    tendencia = np.linspace(-1.5, 0, horas) * rng.uniform(0.5, 1.5)
    ruido_serie = rng.normal(0, ruido, horas)
    serie = temp_atual + tendencia + ruido_serie
    timestamps = [datetime.now() - timedelta(hours=horas - i) for i in range(horas)]
    return timestamps, serie


_ts, _serie = simular_telemetria_temp_externa(20.0)
print(f"TEST: {_serie.mean():.2f} °C")

TEST: 18.97 °C


## Função de avaliação de segurança (GO / NO-GO)

Define o veredito de decolagem com base nos valores.


In [5]:
def avaliar_seguranca(temp_interna, temp_externa_media, estrutura_ok,
                       nivel_energia, pressao_tanque, status_modulos):
    checagens = []

    ok = temp_interna <= TEMP_INTERNA_MAX

    checagens.append(("Temperatura interna", ok,
        f"{temp_interna:.1f} °C (limite: ≤ {TEMP_INTERNA_MAX:.0f} °C)"))

    ok = TEMP_EXTERNA_MIN <= temp_externa_media <= TEMP_EXTERNA_MAX

    checagens.append(("Temperatura externa (média 24h)", ok,
        f"{temp_externa_media:.1f} °C (faixa segura: {TEMP_EXTERNA_MIN:.0f}–{TEMP_EXTERNA_MAX:.0f} °C)"))

    ok = estrutura_ok == 1

    checagens.append(("Integridade estrutural", ok,
        "OK, passou na inspeção da ground crew" if ok else "FALHA estrutural detectada"))

    ok = nivel_energia >= ENERGIA_MINIMA
    checagens.append(("Nível de energia", ok,
        f"{nivel_energia:.0f}% (mínimo necessário: {ENERGIA_MINIMA:.0f}%)"))

    ok = PRESSAO_MIN <= pressao_tanque <= PRESSAO_MAX
    checagens.append(("Pressão do tanque LOX/LCH4", ok,
        f"{pressao_tanque:.0f} psi (faixa segura: {PRESSAO_MIN:.0f}–{PRESSAO_MAX:.0f} psi)"))

    falhas = [nome for nome, status in status_modulos.items() if status == 0]

    ok = len(falhas) == 0
    detalhe = "Todos os módulos OK" if ok else f"FALHA em: {', '.join(falhas)}"
    checagens.append(("Módulos críticos", ok, detalhe))

    geral_ok = all(c[1] for c in checagens)
    return geral_ok, checagens

## Painel interativo

Mexa nos controles abaixo para simular diferentes cenários de telemetria. O checklist e os gráficos são atualizados automaticamente.


In [6]:
titulo_painel = widgets.HTML(
    "<h3>️ Painel de telemetria do Lançamento da Aurora-Siger</h3>"
)

slider_temp_interna = widgets.FloatSlider(
    value=25, min=-20, max=100, step=0.5,
    description='Temp. interna (°C):',
    style={'description_width': 'initial'}, layout=Layout(width='520px'))

slider_temp_externa = widgets.FloatSlider(
    value=20, min=-30, max=55, step=0.5,
    description='Temp. externa atual (°C):',
    style={'description_width': 'initial'}, layout=Layout(width='520px'))

toggle_estrutura = widgets.ToggleButtons(
    options=[('! Falha detectada', 0), ('O Passou na inspeção', 1)],
    value=1, description='Estrutura:',
    style={'description_width': 'initial'})

slider_energia = widgets.IntSlider(
    value=85, min=0, max=100, step=1,
    description='Nível de energia (%):',
    style={'description_width': 'initial'}, layout=Layout(width='520px'))

slider_pressao = widgets.FloatSlider(
    value=325, min=0, max=500, step=1,
    description='Pressão do tanque (psi):',
    style={'description_width': 'initial'}, layout=Layout(width='520px'))

label_modulos = widgets.HTML("<b>Status dos módulos críticos:</b>")
checkboxes_modulos = [
    widgets.Checkbox(value=True, description=nome,
                      style={'description_width': 'initial'},
                      layout=Layout(width='420px'))
    for nome in MODULOS_CRITICOS
]

def atualizar_dashboard(temp_interna, temp_externa_atual, estrutura_ok,
                         nivel_energia, pressao_tanque, **kwargs):
    clear_output(wait=True)

    status_modulos = {
        nome: (1 if kwargs[f'mod_{i}'] else 0)
        for i, nome in enumerate(MODULOS_CRITICOS)
    }

    timestamps, serie_temp = simular_telemetria_temp_externa(temp_externa_atual)
    temp_externa_media = float(np.mean(serie_temp))

    geral_ok, checagens = avaliar_seguranca(
        temp_interna, temp_externa_media, estrutura_ok,
        nivel_energia, pressao_tanque, status_modulos)

    linhas_html = "".join(
        f"<tr><td style='padding:4px 8px;'>{'O' if ok else '!'}</td>"
        f"<td style='padding:4px 8px;'><b>{nome}</b></td>"
        f"<td style='padding:4px 8px;'>{detalhe}</td></tr>"
        for nome, ok, detalhe in checagens
    )

    verdoso = "#3fbc36"
    vermelho = "#bc3636"
    cor = verdoso if geral_ok else vermelho
    texto_status = "GO PARA LANÇAMENTO!!!!!!" if geral_ok else "NO-GO LANÇAMENTO ABORTADO!!!!!!!!!!"

    # kill me

    html = f"""
    <div style="border:2px solid {cor}; border-radius:10px; padding:14px; margin-bottom:10px; font-family:sans-serif;">
        <h2 style="color:{cor}; margin:0;">{texto_status}</h2>
        <p style="margin:4px 0 12px 0; color:#555;">Aurora-Siger, checklist de telemetria pré-lançamento</p>
        <table style="width:100%; border-collapse: collapse;">
        <tr style="text-align:left; border-bottom:1px solid #ccc;"><th></th><th>Item</th><th>Detalhe</th></tr>
        {linhas_html}
        </table>
    </div>
    """
    display(HTML(html))

    fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))

    horas_atras = list(range(24, 0, -1))
    axs[0].plot(horas_atras, serie_temp, marker='o', color='#2b6cb0')
    axs[0].axhspan(TEMP_EXTERNA_MIN, TEMP_EXTERNA_MAX, color=verdoso, alpha=0.15, label='Faixa segura')
    axs[0].axhline(temp_externa_media, color='#2b6cb0', linestyle='--', linewidth=1,
                    label=f'Média 24h: {temp_externa_media:.1f} °C')
    axs[0].set_title('Temp. externa (últimas 24h)')
    axs[0].set_xlabel('Horas atrás')
    axs[0].set_ylabel('°C')
    axs[0].invert_xaxis()
    axs[0].legend(fontsize=8)

    nomes_param = ['Temp.\ninterna (°C)', 'Energia\n(%)', 'Pressão\ntanque (psi)']
    valores = [temp_interna, nivel_energia, pressao_tanque]
    limites = [TEMP_INTERNA_MAX, ENERGIA_MINIMA, PRESSAO_MAX]
    cores_barras = [
        verdoso if checagens[0][1] else vermelho,
        verdoso if checagens[3][1] else vermelho,
        verdoso if checagens[4][1] else vermelho,
    ]
    barras = axs[1].bar(nomes_param, valores, color=cores_barras)
    for i, lim in enumerate(limites):
        axs[1].hlines(lim, i - 0.4, i + 0.4, colors='black', linestyles='dashed', linewidth=1.2)
    axs[1].set_title('Parâmetros atuais vs. limites de segurança\n(linha tracejada é o limite)')

    plt.tight_layout()
    plt.show()


controles = {
    'temp_interna': slider_temp_interna,
    'temp_externa_atual': slider_temp_externa,
    'estrutura_ok': toggle_estrutura,
    'nivel_energia': slider_energia,
    'pressao_tanque': slider_pressao,
}
for i, cb in enumerate(checkboxes_modulos):
    controles[f'mod_{i}'] = cb

saida = interactive_output(atualizar_dashboard, controles)

painel = VBox([
    titulo_painel,
    slider_temp_interna,
    slider_temp_externa,
    toggle_estrutura,
    slider_energia,
    slider_pressao,
    label_modulos,
    *checkboxes_modulos,
])

display(painel, saida)

Output()

## Assistente de Bordo, Borealis

Converse com o assistente de bordo da Aurora-Siger, a Borealis para tirar dúvidas sobre a nave: critérios de segurança, funcionamento dos módulos críticos, ou a leitura atual do painel de telemetria acima.

O assistente roda com o **Qwen via Ollama, localmente na própria máquina do Colab**, então não são respostas prontas, e não precisa de nenhuma chave de API.

Para usar:

1. Rode a célula "Instalar e iniciar o Ollama" abaixo: ela instala o Ollama, sobe o servidor local e baixa o modelo Qwen (pode levar um tempo na primeira vez).
2. Rode a célula seguinte para confirmar que o servidor está pronto.
3. Rode a célula do painel de chat e converse com o assistente no campo de texto.

> Como o modelo roda localmente no runtime do Colab (sem GPU dedicada garantida no plano gratuito), as respostas podem demorar mais do que em uma API na nuvem, mas é melhor pra testar. Se o Colab reiniciar o runtime, é preciso rodar as células de instalação de novo.

In [7]:
!sudo apt install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama

import subprocess, time

subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # dá um tempo para o servidor subir antes de baixar o modelo

!ollama pull qwen2.5:3b

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 52 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (854 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directo

In [8]:
import ollama

MODELO_IA = "qwen2.5:3b"
ollama_pronto = False

def verificar_ollama():
    global ollama_pronto
    try:
        modelos = [m["model"] for m in ollama.list().get("models", [])]
    except Exception as e:
        print(f"Sem resposta do Ollama local ({e}). Rode a célula acima pro setup")
        ollama_pronto = False
        return
    if not any(MODELO_IA in m for m in modelos):
        print(f"Modelo {MODELO_IA} ainda não encontrado localmente. Rode a célula de instalação acima")
        ollama_pronto = False
        return
    ollama_pronto = True
    print("Ollama pronto. Assistente de bordo pronto para uso")

verificar_ollama()

Ollama pronto. Assistente de bordo pronto para uso


### Painel de chat com o assistente

Depois de rodar as células de instalação e verificação do Ollama acima, use o campo abaixo para perguntar coisas como "qual a pressão mínima do tanque?", "o que acontece se um módulo crítico falhar?" ou "como estão os parâmetros agora?".

In [9]:
def montar_bloco_leitura_atual():
    try:
        status_modulos_atual = {
            nome: bool(cb.value) for nome, cb in zip(MODULOS_CRITICOS, checkboxes_modulos)
        }
        modulos_txt = "; ".join(
            f"{nome}: {'OK' if ok else 'FALHA'}" for nome, ok in status_modulos_atual.items()
        )
        return (
            "LEITURA ATUAL DO PAINEL (em tempo real, pode ter mudado desde a última pergunta):\n"
            f"- Temperatura interna: {slider_temp_interna.value} °C\n"
            f"- Temperatura externa (valor do slider, antes da média 24h): {slider_temp_externa.value} °C\n"
            f"- Estrutura: {'OK, passou na inspeção' if toggle_estrutura.value == 1 else 'FALHA detectada'}\n"
            f"- Energia: {slider_energia.value}%\n"
            f"- Pressão do tanque: {slider_pressao.value} psi\n"
            f"- Módulos: {modulos_txt}"
        )
    except NameError:
        return "LEITURA ATUAL DO PAINEL: indisponível (rode a célula do painel interativo acima primeiro)."


def montar_system_prompt():
    return f"""Você é a assistente de bordo (IA chamada Borealis) da nave Aurora-Siger, uma espaçonave fictícia rumo ao Planeta X.
Seu papel é ajudar a tripulação e a ground crew a entender os sistemas da nave e os critérios de
segurança usados na checagem de lançamento, com respostas claras, diretas e curtas.

DADOS DE REFERÊNCIA DA NAVE (use exatamente estes valores, não invente outros):
- Temperatura interna máxima: {TEMP_INTERNA_MAX} °C (limite de operação das baterias)
- Temperatura externa segura (média das últimas 24h): entre {TEMP_EXTERNA_MIN} °C e {TEMP_EXTERNA_MAX} °C
- Energia mínima necessária no lançamento: {ENERGIA_MINIMA}% da capacidade das baterias
- Pressão nominal do tanque LOX/LCH4: {PRESSAO_NOMINAL} psi (faixa aceitável: {PRESSAO_MIN}–{PRESSAO_MAX} psi)
- Módulos críticos monitorados: {", ".join(MODULOS_CRITICOS)}
- Um lançamento só recebe veredito GO se todos os critérios acima forem atendidos ao mesmo tempo

{montar_bloco_leitura_atual()}

Instruções:
- Responda sempre em português.
- Se perguntarem sobre a leitura atual do painel, use os dados de "LEITURA ATUAL DO PAINEL" acima.
- Se a pergunta estiver fora do escopo da nave/lançamento, diga educadamente que seu foco é a missão
  Aurora-Siger, mas tente ajudar brevemente mesmo assim.
- Seja conciso (2-4 frases), a menos que peçam mais detalhes."""

historico_conversa = []

campo_pergunta = widgets.Text(
    placeholder="Digite sua pergunta para o assistente de bordo...",
    layout=Layout(width="500px"))
botao_enviar = widgets.Button(description="Enviar", button_style="primary", icon="paper-plane")
botao_limpar = widgets.Button(description="Limpar")
saida_chat = widgets.Output(
    layout=Layout(border="1px solid #ccc", padding="10px", width="600px",
                  max_height="360px", overflow_y="auto"))

def exibir_mensagem(remetente, texto, cor):
    with saida_chat:
        display(HTML(
            f"<div style='margin:6px 0; font-family:sans-serif;'>"
            f"<b style='color:{cor};'>{remetente}:</b> {texto}</div>"
        ))


def enviar_pergunta(_=None):
    pergunta = campo_pergunta.value.strip()
    if not pergunta:
        return
    campo_pergunta.value = ""

    if not ollama_pronto:
        exibir_mensagem("Sistema", "O servidor Ollama local não está pronto. Rode as células de instalação acima.", "#bc3636")
        return

    exibir_mensagem("Você", pergunta, "#2b6cb0")
    historico_conversa.append({"role": "user", "content": pergunta})

    botao_enviar.disabled = True
    botao_enviar.description = "Consultando..."
    try:
        mensagens = [{"role": "system", "content": montar_system_prompt()}] + historico_conversa
        resposta = ollama.chat(
            model=MODELO_IA,
            messages=mensagens,
        )
        texto_resposta = resposta["message"]["content"].strip()
        if not texto_resposta:
            texto_resposta = "(resposta vazia)"
    except Exception as e:
        texto_resposta = f"Não consegui falar com a IA agora ({e})."
    finally:
        botao_enviar.disabled = False
        botao_enviar.description = "Enviar"

    historico_conversa.append({"role": "assistant", "content": texto_resposta})
    exibir_mensagem("Borealis", texto_resposta.replace(chr(10), "<br>"), "#3fbc36")

def limpar_conversa(_=None):
    historico_conversa.clear()
    saida_chat.clear_output()


botao_enviar.on_click(enviar_pergunta)
if hasattr(campo_pergunta, "on_submit"):
    campo_pergunta.on_submit(enviar_pergunta)
botao_limpar.on_click(limpar_conversa)

display(widgets.HTML("<h3>Borealis</h3>"))
display(VBox([HBox([campo_pergunta, botao_enviar, botao_limpar]), saida_chat]))

HTML(value='<h3>Borealis</h3>')

## Notas e possíveis melhorias

- Todos os valores de referências podem ser alterados sem mudar o resto do código, no bloco de Valores de Referência.
- Como é uma simulação simplificada, a "telemetria" de temperatura externa é gerada sinteticamente em torno do valor do slider. Para um projeto mais avançado, essa série poderia vir de um arquivo de uma API de clima, ou um arquivo ou qualquer outra fonte melhor.
- Ideias pra melhorar depois
  - Fazer um log de tentativas de lançamento.
  - Adicionar um botão "Lançar que só fica habilitado quando o status geral é GO.